In [0]:
from pyspark.sql import functions as F, Window

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

df = spark.table("workspace.bronze.tb_movies_info")
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- tconst: string (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- status: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



In [0]:
window_dedup = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_dedup = (df
    .withColumn("rn", F.row_number().over(window_dedup))
    .filter(F.col("rn") == 1)
    .drop("rn"))

print(f"Antes: {df.count()} | Depois do dedup: {df_dedup.count()}")

Antes: 427720 | Depois do dedup: 97879


In [0]:
df_renamed = (df_dedup
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento_raw")
    .withColumnRenamed("runtime", "duracao_minutos_raw")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_raw")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao"))

In [0]:
df_status = df_renamed.withColumn(
    "status_normalizado",
    F.upper(F.trim(F.regexp_replace(F.col("status_raw"), r"[-_]+", " ")))
).withColumn(
    "status_normalizado",
    F.trim(F.regexp_replace(F.col("status_normalizado"), r"\s+", " "))
)

status_map = {
    "RELEASED": "Lançado",
    "POST PRODUCTION": "Pós-Produção",
    "IN PRODUCTION": "Em Produção",
    "PLANNED": "Planejado",
    "RUMORED": "Rumores",
    "CANCELED": "Cancelado",
    "CANCELLED": "Cancelado",
}
mapping_expr = F.create_map([F.lit(x) for pair in status_map.items() for x in pair])

df_status_final = df_status.withColumn(
    "status_filme",
    F.coalesce(mapping_expr[F.col("status_normalizado")], F.lit("Não Informado"))
).drop("status_raw", "status_normalizado")

df_status_final.groupBy("status_filme").count().show()

+-------------+-----+
| status_filme|count|
+-------------+-----+
|      Lançado|96463|
| Pós-Produção|  701|
|    Planejado|   47|
|Não Informado|   64|
|  Em Produção|  604|
+-------------+-----+



In [0]:
from pyspark.sql import functions as F

df_status_final.select("data_lancamento_raw").distinct().orderBy(F.rand()).show(30, truncate=False)

+-------------------+
|data_lancamento_raw|
+-------------------+
|07-20-2023         |
|2021-06-12         |
|2020-01-25         |
|10-13-2021         |
|14/11/2020         |
|25/10/2017         |
|03/06/2022         |
|2021-04-20         |
|30/09/2022         |
|11-12-2019         |
|2023-11-11         |
|12-01-2019         |
|2019-01-04         |
|30/09/2017         |
|2020-09-20         |
|2019-12-23         |
|2019-06-24         |
|11/01/2019         |
|2016-04-23         |
|01/10/2018         |
|2019-04-20         |
|02-06-2016         |
|17/10/2019         |
|09-28-2018         |
|08/06/2018         |
|07-08-2021         |
|2023-02-11         |
|2019-04-29         |
|05-29-2016         |
|30/09/2021         |
+-------------------+
only showing top 30 rows


In [0]:
df_date = df_status_final.withColumn(
    "data_lancamento",
    F.coalesce(
        F.try_to_date("data_lancamento_raw", "yyyy-MM-dd"),
        F.try_to_date("data_lancamento_raw", "MM-dd-yyyy"),
        F.try_to_date("data_lancamento_raw", "dd/MM/yyyy")
    )
).withColumn(
    "duracao_minutos", F.expr("try_cast(duracao_minutos_raw AS INT)")
).withColumn(
    "ano_lancamento", F.year(F.col("data_lancamento"))
).drop("data_lancamento_raw", "duracao_minutos_raw")

total = df_date.count()
nulas = df_date.filter(F.col("data_lancamento").isNull()).count()
print(f"Total: {total} | Datas NULL após conversão: {nulas} ({nulas/total:.2%})")

Total: 97879 | Datas NULL após conversão: 64 (0.07%)


In [0]:
df_date.filter(F.col("data_lancamento").isNull()).select("id_filme", "titulo").show(10, truncate=False)

+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
df_final_info = df_date.select(
    "id_filme",
    "titulo",
    "titulo_original",
    "data_lancamento",
    "duracao_minutos",
    "idioma_original",
    "status_filme",
    F.regexp_replace(F.col("sinopse"), r"^\\+", "").alias("sinopse"),
    "frase_divulgacao",
    "ano_lancamento"
)

(df_final_info.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.tb_info_filmes"))

print(f"OK: workspace.silver.tb_info_filmes -> {df_final_info.count()} linhas")
display(df_final_info.limit(10))

OK: workspace.silver.tb_info_filmes -> 97879 linhas


id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
66213,Iruttu Araiyil Murattu Kuthu,இருட்டு அறையில் முரட்டு குத்து,2018-05-04,118,ta,Lançado,A sexually-starved spirit traps two couples in a bungalow in Bangkok where they have planned to stay for a week. The spirit wants one of the two men to have sex with her. Can they manage to escape?,null,2018
68730,Silence,Silence,2016-12-22,161,en,Lançado,"Two Jesuit priests travel to seventeenth century Japan which has, under the Tokugawa shogunate, banned Catholicism and almost all foreign contact.",null,2016
80765,Dimmu Borgir - Forces of the Northern Night,Dimmu Borgir - Forces of the Northern Night,2017-04-28,59,en,Lançado,01. Xibir 02. Born Treacherous 03. Gateways 04. Dimmu Borgir (orchestral) 05. Dimmu Borgir 06. Ritualist 07. A Jewel Traced Through Coal 08. Eradication Instincts Defined (orchestral) 09. Vredesbyrd 10. Progenies Of The Great Apocalypse 11. The Serpentine Offering 12. Fear And Wonder (orchestral) 13. Puritania 14. Kings Of The Carnival Creation 15. Mourning Palace 16. Perfection Or Vanity,null,2017
115436,Hason Raja,Hason Raja,2017-03-31,165,bn,Lançado,"The story of Hason Raja, a flamboyant, ruthless zamindar from Sylhet who fell in love with Dilaram, who transformed him and later he became a poet, singing songs and wandering all across Bangladesh.",null,2017
137116,Smurfs: The Lost Village,Smurfs: The Lost Village,2017-03-23,89,en,Lançado,"In this fully animated, all-new take on the Smurfs, a mysterious map sets Smurfette and her friends Brainy, Clumsy and Hefty on an exciting race through the Forbidden Forest leading to the discovery of the biggest secret in Smurf history.",null,2017
166426,Pirates of the Caribbean: Dead Men Tell No Tales,Pirates of the Caribbean: Dead Men Tell No Tales,2017-05-23,128,en,Lançado,"Thrust into an all-new adventure, a down-on-his-luck Capt. Jack Sparrow feels the winds of ill-fortune blowing even more strongly when deadly ghost sailors led by his old nemesis, the evil Capt. Salazar, escape from the Devil's Triangle. Jack's only hope of survival lies in seeking out the legendary Trident of Poseidon, but to find it, he must forge an uneasy alliance with a brilliant and beautiful astronomer and a headstrong young man in the British navy.",null,2017
181808,Star Wars: The Last Jedi,Star Wars: The Last Jedi,2017-12-13,152,en,Lançado,"Rey develops her newly discovered abilities with the guidance of Luke Skywalker, who is unsettled by the strength of her powers. Meanwhile, the Resistance prepares to do battle with the First Order.",Darkness rises... and light to meet it,2017
196712,Edge,Edge,2018-08-03,76,en,Lançado,"EDGE. On the surface, it's the story of a madman terrorizing a helpless city, taking lives without a second thought. Blood is this man's water. Enter Jack Rivers, detective for the city's police department and ace headhunter who always gets his man. Stymied by the difficulty of catching this serial murderer, Jack begins to stumble in a blind fury and the line between 'justice' and 'murder' becomes blurred. This is the story of a man whose life is about to change forever. He's about to go over the EDGE.",He's about to go over the EDGE.,2018
245842,The King's Daughter,The King's Daughter,2022-01-21,94,en,Lançado,"King Louis XIV's quest for immortality leads him to capture and steal a mermaid's life force, a move that is further complicated by his illegitimate daughter's discovery of the creature.",What on Earth can hold more power than a king?,2022
248412,Western X,Western X,2016-06-01,90,en,Lançado,A man struggles to find out the truth behind his identity while fighting an evil army.,null,2016


In [0]:
df_fin = spark.table("workspace.bronze.tb_movies_financials")

window_dedup_fin = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_fin_dedup = (df_fin
    .withColumn("rn", F.row_number().over(window_dedup_fin))
    .filter(F.col("rn") == 1)
    .drop("rn"))

df_fin_dedup.select("budget").distinct().filter(~F.col("budget").rlike(r"^\d+(\.\d+)?$")).show(20, truncate=False)

+-------------+
|budget       |
+-------------+
|90.0M        |
|USD 30000000 |
|84.0M        |
|$ 11795877   |
|5.0M         |
|USD 5000000  |
|$ 20000000   |
|$ 250000000  |
|$ 80000000   |
|45.0M        |
|$ 185000000  |
|15.0M        |
|300.0K       |
|USD 3500     |
|N/A          |
|USD 175000000|
|USD 125000000|
|35.0M        |
|USD 8000000  |
|USD 170000000|
+-------------+
only showing top 20 rows


In [0]:
def parse_money(colname):
    raw_upper = F.upper(F.trim(F.col(colname)))
    
    is_null_text = raw_upper.isin("UNKNOWN", "NÃO INFORMADO", "N/A", "NA", "")
    
    cleaned = F.regexp_replace(raw_upper, r"(USD|\$|R\$|,|\s)", "")
    
    numeric_part = F.regexp_extract(cleaned, r"^(\d+(?:\.\d+)?)", 1)
    numeric_val = F.when(numeric_part == "", None).otherwise(numeric_part.cast("double"))
    
    multiplier = (
        F.when(cleaned.rlike(r"M$"), F.lit(1_000_000.0))
         .when(cleaned.rlike(r"K$"), F.lit(1_000.0))
         .otherwise(F.lit(1.0))
    )
    
    value = F.when(is_null_text, F.lit(None).cast("double")).otherwise(numeric_val * multiplier)
    
    return F.when(value <= 0, F.lit(None).cast("double")).otherwise(value)

df_fin_parsed = (df_fin_dedup
    .withColumnRenamed("id", "id_filme")
    .withColumn("orcamento_usd", parse_money("budget"))
    .withColumn("receita_usd", parse_money("revenue")))

total = df_fin_parsed.count()
nulos_orc = df_fin_parsed.filter(F.col("orcamento_usd").isNull()).count()
nulos_rec = df_fin_parsed.filter(F.col("receita_usd").isNull()).count()
print(f"Total: {total} | orcamento_usd NULL: {nulos_orc} ({nulos_orc/total:.2%}) | receita_usd NULL: {nulos_rec} ({nulos_rec/total:.2%})")

df_fin_parsed.select("id_filme", "budget", "orcamento_usd", "revenue", "receita_usd").show(15, truncate=False)

Total: 99006 | orcamento_usd NULL: 90835 (91.75%) | receita_usd NULL: 95721 (96.68%)
+--------+-------------+-------------+---------+------------+
|id_filme|budget       |orcamento_usd|revenue  |receita_usd |
+--------+-------------+-------------+---------+------------+
|14564   |25000000     |2.5E7        |83080890 |8.308089E7  |
|32471   |0            |NULL         |0        |NULL        |
|38258   |7.5M         |7500000.0    |0        |NULL        |
|38492   |0            |NULL         |0        |NULL        |
|38700   |90.0M        |9.0E7        |426505244|4.26505244E8|
|42018   |0            |NULL         |0        |NULL        |
|42330   |0            |NULL         |0        |NULL        |
|43074   |144000000    |1.44E8       |229147509|2.29147509E8|
|45033   |337200       |337200.0     |0        |NULL        |
|46983   |0            |NULL         |0        |NULL        |
|47933   |USD 165000000|1.65E8       |389681935|3.89681935E8|
|49046   |20000000     |2.0E7        |0        

In [0]:
cotacao_atual = (spark.table("workspace.bronze.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc())
    .limit(1)
    .select("cotacaoCompra")
    .collect()[0]["cotacaoCompra"])

print(f"Cotação usada (mais recente): {cotacao_atual}")

df_fin_final = (df_fin_parsed
    .withColumn("orcamento_brl", F.col("orcamento_usd") * F.lit(cotacao_atual))
    .withColumn("receita_brl", F.col("receita_usd") * F.lit(cotacao_atual))
    .withColumn(
        "lucro_usd",
        F.when(F.col("orcamento_usd").isNotNull() & F.col("receita_usd").isNotNull(),
               F.col("receita_usd") - F.col("orcamento_usd"))
    )
    .withColumn(
        "lucro_brl",
        F.when(F.col("orcamento_brl").isNotNull() & F.col("receita_brl").isNotNull(),
               F.col("receita_brl") - F.col("orcamento_brl"))
    )
    .withColumn(
        "margem_lucro_percentual",
        F.when(F.col("orcamento_usd").isNotNull() & (F.col("orcamento_usd") != 0) & F.col("lucro_usd").isNotNull(),
               F.round(F.col("lucro_usd") / F.col("orcamento_usd") * 100, 2))
    )
)

df_fin_final.select(
    "id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_percentual"
).filter(F.col("orcamento_usd").isNotNull()).show(10, truncate=False)

Cotação usada (mais recente): 5.1569
+--------+-------------+------------+------------------+--------------------+------------+--------------------+-----------------------+
|id_filme|orcamento_usd|receita_usd |orcamento_brl     |receita_brl         |lucro_usd   |lucro_brl           |margem_lucro_percentual|
+--------+-------------+------------+------------------+--------------------+------------+--------------------+-----------------------+
|38700   |9.0E7        |4.26505244E8|4.64121E8         |2.1994448927836003E9|3.36505244E8|1.7353238927836003E9|373.89                 |
|45033   |337200.0     |NULL        |1738906.6800000002|NULL                |NULL        |NULL                |NULL                   |
|68730   |4.6E7        |NULL        |2.372174E8        |NULL                |NULL        |NULL                |NULL                   |
|91639   |7800000.0    |NULL        |4.022382E7        |NULL                |NULL        |NULL                |NULL                   |
|121856  |1

In [0]:
df_final_fin = df_fin_final.select(
    "id_filme",
    "orcamento_usd",
    "receita_usd",
    "orcamento_brl",
    "receita_brl",
    "lucro_usd",
    "lucro_brl",
    "margem_lucro_percentual"
)

(df_final_fin.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.tb_financeiro_filmes"))

print(f"OK: workspace.silver.tb_financeiro_filmes -> {df_final_fin.count()} linhas")
display(df_final_fin.limit(10))

OK: workspace.silver.tb_financeiro_filmes -> 99006 linhas


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
38700,9.0E7,4.26505244E8,4.64121E8,2.1994448927836003E9,3.36505244E8,1.7353238927836003E9,373.89
50022,null,1.8850674E7,null,9.721104075060001E7,null,null,null
66534,null,null,null,null,null,null,null
67326,null,null,null,null,null,null,null
75594,199000.0,null,1026223.1000000001,null,null,null,null
77027,null,null,null,null,null,null,null
80765,null,null,null,null,null,null,null
91639,7800000.0,null,4.022382E7,null,null,null,null
101736,null,null,null,null,null,null,null
109453,null,null,null,null,null,null,null


In [0]:
df_metrics = spark.table("workspace.bronze.tb_movies_metrics")

window_dedup_metrics = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_metrics_dedup = (df_metrics
    .withColumn("rn", F.row_number().over(window_dedup_metrics))
    .filter(F.col("rn") == 1)
    .drop("rn"))

for col in ["popularity", "vote_average", "vote_count", "averageRating", "numVotes"]:
    print(f"\n===== {col} =====")
    df_metrics_dedup.select(col).distinct().filter(
        ~F.col(col).rlike(r"^-?\d+([.,]\d+)?$")
    ).show(15, truncate=False)


===== popularity =====
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|popularity                                                                                                                                                                                                                                     |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Sabine gets serious. She gives Bernd an ultimatum. Because one thing is clear to them: just continuing like this is out of the question."                                                                                                      |
|a dange

In [0]:
df_metrics_renamed = df_metrics_dedup.withColumnRenamed("id", "id_filme")

df_metrics_step1 = df_metrics_renamed.withColumn(
    "popularidade_raw", F.regexp_replace(F.trim(F.col("popularity")), ",", ".")
).withColumn(
    "popularidade", F.expr("try_cast(popularidade_raw AS DOUBLE)")
)

df_metrics_step2 = (df_metrics_step1
    .withColumn("nota_media_tmdb", F.expr("try_cast(vote_average AS DOUBLE)"))
    .withColumn("qtd_votos_tmdb", F.expr("try_cast(vote_count AS INT)"))
    .withColumn("nota_media_imdb", F.expr("try_cast(averageRating AS DOUBLE)"))
    .withColumn("qtd_votos_imdb", F.expr("try_cast(numVotes AS INT)"))
)

df_metrics_final = (df_metrics_step2
    .withColumn("nota_media_tmdb",
        F.when((F.col("nota_media_tmdb") < 0) | (F.col("nota_media_tmdb") > 10), None)
         .otherwise(F.col("nota_media_tmdb")))
    .withColumn("nota_media_imdb",
        F.when((F.col("nota_media_imdb") < 0) | (F.col("nota_media_imdb") > 10), None)
         .otherwise(F.col("nota_media_imdb")))
    .withColumn("qtd_votos_tmdb",
        F.when(F.col("qtd_votos_tmdb") < 0, None).otherwise(F.col("qtd_votos_tmdb")))
    .withColumn("qtd_votos_imdb",
        F.when(F.col("qtd_votos_imdb") < 0, None).otherwise(F.col("qtd_votos_imdb")))
    .withColumn("popularidade",
        F.when(
            (F.col("popularidade") < 0) |
            ((F.col("popularidade") >= 1900) & (F.col("popularidade") <= 2030) & (F.col("popularidade") == F.floor(F.col("popularidade")))),
            None
        ).otherwise(F.col("popularidade")))
)

total = df_metrics_final.count()
for c in ["popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"]:
    nulos = df_metrics_final.filter(F.col(c).isNull()).count()
    print(f"{c}: {nulos} nulos ({nulos/total:.2%})")

popularidade: 4172 nulos (4.21%)
nota_media_tmdb: 3519 nulos (3.55%)
qtd_votos_tmdb: 8384 nulos (8.47%)
nota_media_imdb: 13197 nulos (13.33%)
qtd_votos_imdb: 11692 nulos (11.81%)


In [0]:
df_final_metrics = df_metrics_final.select(
    "id_filme",
    "popularidade",
    "nota_media_tmdb",
    "qtd_votos_tmdb",
    "nota_media_imdb",
    "qtd_votos_imdb"
)

(df_final_metrics.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.tb_metricas_engajamento"))

print(f"OK: workspace.silver.tb_metricas_engajamento -> {df_final_metrics.count()} linhas")
display(df_final_metrics.limit(10))

OK: workspace.silver.tb_metricas_engajamento -> 99013 linhas


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
50022,12.698,5.854,417,null,31025
66534,1.837,null,9,5.8,208
77027,1.475,6.0,1,8.2,180
91639,3.508,5.7,11,5.3,562
124346,2.018,8.5,2,null,177
130421,2.69,3.4,5,null,263
140300,null,6.878,5327,null,null
142611,3.524,7.7,30,7.8,3870
153518,24.75,6.205,3369,null,120814
171782,0.607,7.5,2,7.1,107


In [0]:
df_reviews = spark.table("workspace.bronze.tb_movies_reviews")

df_reviews_renamed = (df_reviews
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario"))

df_reviews_dedup = df_reviews_renamed.dropDuplicates(
    ["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"]
)

df_reviews_step1 = df_reviews_dedup.withColumn(
    "nota_usuario",
    F.when((F.col("nota_usuario") < 0) | (F.col("nota_usuario") > 10), None)
     .otherwise(F.col("nota_usuario"))
)

df_reviews_final = df_reviews_step1.withColumn(
    "comentario_usuario",
    F.when(
        F.col("comentario_usuario").isNull() | (F.trim(F.col("comentario_usuario")) == ""),
        F.lit("Sem comentário")
    ).otherwise(F.col("comentario_usuario"))
)

total = df_reviews_final.count()
print(f"Antes do dedup: {df_reviews_renamed.count()} | Depois: {total}")
df_reviews_final.filter(F.col("comentario_usuario") == "Sem comentário").count()

Antes do dedup: 162060 | Depois: 32412


7776

In [0]:
df_final_reviews = df_reviews_final.select(
    "id_filme",
    "nome_usuario",
    "nota_usuario",
    "comentario_usuario"
)

(df_final_reviews.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.tb_avaliacoes_usuarios"))

print(f"OK: workspace.silver.tb_avaliacoes_usuarios -> {df_final_reviews.count()} linhas")
display(df_final_reviews.limit(10))

OK: workspace.silver.tb_avaliacoes_usuarios -> 32412 linhas


id_filme,nome_usuario,nota_usuario,comentario_usuario
846452,Sérgio Freitas,6.1,Assisti até o final mas não me marcou.
1155082,Carolina Almeida 653,3.2,"Fraco, não recomendo."
923930,Carlos Monteiro 465,9.0,Excepcional! História envolvente e atuações impecáveis.
661409,Matheus Gomes 340,9.8,Perfeito! Um dos melhores filmes que já vi.
447163,Natália Freitas 402,9.1,"Obra-prima do cinema, simplesmente espetacular."
991019,Daniel Cardoso 234,0.5,"Que filme ruim, não assistam."
1176825,Renata Ferreira 589,2.1,Sem comentário
662277,Henrique Ferreira 463,9.2,"Adorei cada minuto, uma experiência inesquecível."
842679,Mariana Lima 361,1.1,Péssimo em todos os sentidos.
1158385,Marcelo Cavalcanti 439,2.6,Horrível! Perda de tempo.


In [0]:
df_credits = spark.table("workspace.bronze.tb_credits_and_tags")

window_dedup_credits = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())
df_credits_dedup = (df_credits
    .withColumn("rn", F.row_number().over(window_dedup_credits))
    .filter(F.col("rn") == 1)
    .drop("rn"))

df_credits_dedup.select("genres").distinct().orderBy(F.rand()).show(20, truncate=False)

+-----------------------------------------------------------------------------------------------------------------+
|genres                                                                                                           |
+-----------------------------------------------------------------------------------------------------------------+
|the original model of coexistence of bees and people                                                             |
|full of controversial moments                                                                                    |
|Science Fiction, Horror, Action                                                                                  |
|Comedy, Romance, Drama, TV Movie                                                                                 |
|Horror; Science Fiction; Mystery; Thriller                                                                       |
|Drama, Romance, Documentary                                            

In [0]:
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery",
    "Romance", "Science Fiction", "TV Movie", "Thriller", "War", "Western"
]

df_genres_split = df_credits_dedup.withColumn(
    "genero_individual",
    F.explode(F.split(F.col("genres"), r"\s*[,;|]\s*"))
).withColumn(
    "genero_individual", F.trim(F.col("genero_individual"))
)

df_generos_final = (df_genres_split
    .filter(F.lower(F.col("genero_individual")).isin([g.lower() for g in generos_validos]))
    .select(
        F.col("id").alias("id_filme"),
        F.initcap(F.col("genero_individual")).alias("nome_genero")
    )
    .withColumn("nome_genero", F.regexp_replace(F.col("nome_genero"), "^Tv Movie$", "TV Movie"))
    .dropDuplicates(["id_filme", "nome_genero"])
)

print(f"Total de relações filme-gênero: {df_generos_final.count()}")
df_generos_final.groupBy("nome_genero").count().orderBy(F.desc("count")).show(20, truncate=False)

Total de relações filme-gênero: 141947
+---------------+-----+
|nome_genero    |count|
+---------------+-----+
|Drama          |32646|
|Documentary    |19233|
|Comedy         |18821|
|Thriller       |10400|
|Horror         |9853 |
|Romance        |7712 |
|Action         |6100 |
|Crime          |4781 |
|Animation      |4516 |
|TV Movie       |4112 |
|Science Fiction|3808 |
|Family         |3761 |
|Mystery        |3355 |
|Fantasy        |3306 |
|Adventure      |2889 |
|Music          |2824 |
|History        |2439 |
|War            |974  |
|Western        |417  |
+---------------+-----+



In [0]:
(df_generos_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.tb_generos"))

print(f"OK: workspace.silver.tb_generos -> {df_generos_final.count()} linhas")
display(df_generos_final.limit(10))

OK: workspace.silver.tb_generos -> 141947 linhas


id_filme,nome_genero
42330,Animation
42330,Fantasy
42330,Adventure
49046,Drama
49046,War
50022,Comedy
50022,Crime
50022,Mystery
55341,Horror
55341,Mystery


In [0]:
def extrair_entidades(df, coluna_origem, tipo_fixo):
    return (df
        .withColumn("entidade_individual", F.explode(F.split(F.col(coluna_origem), r"\s*[,;|]\s*")))
        .withColumn("entidade_individual", F.trim(F.col("entidade_individual")))
        .filter(F.col("entidade_individual") != "")
        .select(
            F.col("id").alias("id_filme"),
            F.initcap(F.col("entidade_individual")).alias("nome_pessoa_empresa"),
            F.lit(tipo_fixo).alias("tipo_entidade")
        )
    )

df_atores = extrair_entidades(df_credits_dedup, "cast", "Ator")
df_diretores = extrair_entidades(df_credits_dedup, "directors", "Diretor")
df_roteiristas = extrair_entidades(df_credits_dedup, "writers", "Roteirista")
df_produtoras = extrair_entidades(df_credits_dedup, "production_companies", "Produtora")

df_pessoas_empresas_raw = (df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras))

print(f"Total antes de filtrar lixo: {df_pessoas_empresas_raw.count()}")
df_pessoas_empresas_raw.groupBy("tipo_entidade").count().show()

df_pessoas_empresas_raw.filter(F.length("nome_pessoa_empresa") > 60).show(15, truncate=False)

Total antes de filtrar lixo: 910758
+-------------+------+
|tipo_entidade| count|
+-------------+------+
|         Ator|544343|
|      Diretor|108083|
|   Roteirista|137478|
|    Produtora|120854|
+-------------+------+

+--------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+
|id_filme|nome_pessoa_empresa                                                                                                                                                                                                                                                                             |tipo_entidade|
+--------+---------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
stopwords_prosa = [
    "the", "and", "who", "with", "from", "that", "this", "her", "his",
    "was", "were", "has", "have", "will", "she", "him", "they", "their",
    "when", "where", "which", "would", "could", "should", "not", "but",
    "into", "than", "then", "over", "such", "each", "more", "most"
]
stopwords_regex = r"(?i)\b(" + "|".join(stopwords_prosa) + r")\b"

idiomas_conhecidos = [
    "english", "french", "german", "spanish", "italian", "portuguese",
    "japanese", "korean", "chinese", "mandarin", "cantonese", "russian",
    "arabic", "hindi", "dutch", "swedish", "norwegian", "danish", "polish",
    "turkish", "greek", "hebrew", "thai", "vietnamese", "indonesian"
]

def entidade_valida(col):
    return (
        (F.length(F.trim(col)) <= 60) &                          
        (F.size(F.split(F.trim(col), r"\s+")) <= 5) &           
        (~F.trim(col).rlike(stopwords_regex)) &                 
        (~F.trim(col).rlike(r"[\"'\\]")) &
        (~F.trim(col).rlike(r"^/")) &
        (F.trim(col).rlike(r"[A-Za-zÀ-ÿ]")) &                     
        (~F.lower(F.trim(col)).isin(idiomas_conhecidos))        
    )

def extrair_entidades(df, coluna_origem, tipo_fixo):
    return (df
        .withColumn("entidade_individual", F.explode(F.split(F.col(coluna_origem), r"\s*[,;|]\s*")))
        .withColumn("entidade_individual", F.trim(F.col("entidade_individual")))
        .filter(F.col("entidade_individual") != "")
        .filter(entidade_valida(F.col("entidade_individual")))
        .select(
            F.col("id").alias("id_filme"),
            F.initcap(F.col("entidade_individual")).alias("nome_pessoa_empresa"),
            F.lit(tipo_fixo).alias("tipo_entidade")
        )
    )

df_atores = extrair_entidades(df_credits_dedup, "cast", "Ator")
df_diretores = extrair_entidades(df_credits_dedup, "directors", "Diretor")
df_roteiristas = extrair_entidades(df_credits_dedup, "writers", "Roteirista")
df_produtoras = extrair_entidades(df_credits_dedup, "production_companies", "Produtora")

df_pessoas_empresas_final = (df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
    .dropDuplicates(["id_filme", "nome_pessoa_empresa", "tipo_entidade"]))

print(f"Total após filtro: {df_pessoas_empresas_final.count()}")
df_pessoas_empresas_final.groupBy("tipo_entidade").count().show()

df_pessoas_empresas_final.filter(F.length("nome_pessoa_empresa") > 45).show(10, truncate=False)

# nomes geralmente são curtos e não contêm palavras de ligação, por isso escolhi essa abordagem acima

Total após filtro: 879835
+-------------+------+
|tipo_entidade| count|
+-------------+------+
|    Produtora|113862|
|   Roteirista|129340|
|      Diretor|100050|
|         Ator|536583|
+-------------+------+

+--------+--------------------------------------------------+-------------+
|id_filme|nome_pessoa_empresa                               |tipo_entidade|
+--------+--------------------------------------------------+-------------+
|240632  |Centro Costarricense De Producción Cinematográfica|Produtora    |
|654122  |Cinémathèque De La Fédération Wallonie-bruxelles  |Produtora    |
|337556  |Geißendörfer Film- Und Fernsehproduktion (gff)    |Produtora    |
|612701  |Levenstein Harbour Longino Televised Theatricals  |Produtora    |
|806792  |Mgc Producciones Cinematográficas Y Audiovisuales |Produtora    |
|510265  |Tele München Fernseh Produktionsgesellschaft (tmg)|Produtora    |
|731536  |Medien- Und Filmgesellschaft Baden-württemberg    |Produtora    |
|271404  |Hongmaisui Internat

In [0]:
(df_pessoas_empresas_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.tb_pessoas_empresas"))

print(f"OK: workspace.silver.tb_pessoas_empresas -> {df_pessoas_empresas_final.count()} linhas")
display(df_pessoas_empresas_final.limit(10))



OK: workspace.silver.tb_pessoas_empresas -> 879835 linhas


id_filme,nome_pessoa_empresa,tipo_entidade
320590,Dagoberto Gama,Ator
345009,Luka Peroš,Ator
374475,Sandra Hüller,Ator
376867,Alex R. Hibbert,Ator
381355,Tarusuke Shingaki,Ator
398392,Nicolas Chupin,Ator
401441,Mia Freedman,Ator
403820,Martin W. Payne,Ator
263341,Jason Scott Lee,Ator
270854,John B. Woods,Ator


In [0]:
df_cotacao_raw = (spark.table("workspace.bronze.tb_cotacao_dolar")
    .withColumn("data_cotacao", F.to_date(F.col("dataHoraCotacao")))
    .dropDuplicates(["data_cotacao", "cotacaoCompra"]))

min_max = df_cotacao_raw.agg(F.min("data_cotacao"), F.max("data_cotacao")).collect()[0]
data_min, data_max = min_max[0], min_max[1]
print(f"Intervalo: {data_min} até {data_max}")

df_calendario = spark.sql(f"""
    SELECT explode(sequence(to_date('{data_min}'), to_date('{data_max}'), interval 1 day)) AS data_cotacao
""")

df_cotacao_join = (df_calendario
    .join(df_cotacao_raw.select("data_cotacao", "cotacaoCompra"), on="data_cotacao", how="left")
    .orderBy("data_cotacao"))

window_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)

df_cotacao_final = df_cotacao_join.withColumn(
    "cotacao_dolar",
    F.last("cotacaoCompra", ignorenulls=True).over(window_ffill)
).select("data_cotacao", "cotacao_dolar")

total = df_cotacao_final.count()
print(f"Total de linhas: {total}")
df_cotacao_final.show(20, truncate=False)

Intervalo: 2026-09-14 até 2026-09-18


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total de linhas: 5
+------------+-------------+
|data_cotacao|cotacao_dolar|
+------------+-------------+
|2026-09-14  |5.169        |
|2026-09-15  |5.1484       |
|2026-09-16  |5.152        |
|2026-09-17  |5.1515       |
|2026-09-18  |5.1569       |
+------------+-------------+



In [0]:
(df_cotacao_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.tb_cotacao_dolar"))

print(f"OK: workspace.silver.tb_cotacao_dolar -> {df_cotacao_final.count()} linhas")
display(df_cotacao_final)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


OK: workspace.silver.tb_cotacao_dolar -> 5 linhas


data_cotacao,cotacao_dolar
2026-09-14,5.169
2026-09-15,5.1484
2026-09-16,5.152
2026-09-17,5.1515
2026-09-18,5.1569
